# Solo-output grokking

Trains the model on a **single output wire** at a time (loss masked to that
wire; eval still records all 256 outputs) and compares against the same
wire's trajectory in the multi-task run. Predictions being tested:

1. A single output shows a grokking **step** (plateau at chance -> cliff),
   not a power law — the smooth scaling curve exists only in aggregate.
2. Solo transitions come **later** than multi-task ones (no curriculum from
   easy outputs building shared features), with the gap growing in tap depth.
3. Pure-parity outputs (wires 42 and 5 of circuit seed 0) are the cleanest
   single steps and the most seed-sensitive.

Runs GPU-friendly on Colab (auto-detected below) or locally; every run is
resumable and skipped once complete.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

## Target selection

Parity controls (wires 42, 5 — pure XORs found in the analysis) plus, per tap
depth in `DEPTHS`, the first `PER_DEPTH` wires that the multi-task reference
run actually solved (final acc > 0.95), so the solo-vs-joint comparison is
about *when*, not *whether*.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np

from train import RunConfig, load_run, run

# --- experiment config ---
WIDTH, MLP_DEPTH, LR = 360, 7, 1e-3   # must match an existing multi-task run
STEPS = 50_000
SEEDS = [0, 1]
DEPTHS = [2, 3, 4]
PER_DEPTH = 2
PARITY_WIRES = [42, 5]                # pure 3- and 4-parity (circuit seed 0)
OUT_DIR = "runs/solo"

ref_path = glob.glob(f"runs/w{WIDTH}_d{MLP_DEPTH}_*_s{STEPS}_cs0_ms0.npz")
assert ref_path, "matching multi-task reference run not found in runs/"
ref_cfg, ref = load_run(ref_path[0])
depths = ref["out_depths"]
final_acc = ref["per_out_acc"][-1]

targets = list(PARITY_WIRES)
for k in DEPTHS:
    ok = np.flatnonzero((depths == k) & (final_acc > 0.95))
    targets += [w for w in ok[:PER_DEPTH] if w not in targets]

print(f"{'wire':>5s} {'tap depth':>9s} {'multi-task acc':>14s}")
for w in targets:
    print(f"{w:>5d} {depths[w]:>9d} {final_acc[w]:>14.3f}")

configs = [
    RunConfig(width=WIDTH, mlp_depth=MLP_DEPTH, lr=LR, steps=STEPS,
              output_wires=(int(w),), model_seed=s, out_dir=OUT_DIR)
    for w in targets for s in SEEDS
]
print(f"\n{len(configs)} runs (~{len(configs) * 6} GPU-min at this shape)")

## Run (idempotent — interrupt and re-run freely)

In [ ]:
for cfg in configs:
    run(cfg)

## Solo vs multi-task trajectories

Solid: solo runs (one per seed). Dashed black: the same wire in the
multi-task run. Log-log; chance dotted.

In [ ]:
CHANCE = np.log(2)
ncols = 4
nrows = -(-len(targets) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows),
                         sharex=True, sharey=True)
seed_cols = plt.cm.tab10(np.arange(len(SEEDS)))

solo = {}
for cfg in configs:
    if cfg.npz_path.exists():
        solo[(cfg.output_wires[0], cfg.model_seed)] = load_run(cfg.npz_path)[1]

for ax, w in zip(axes.flat, targets):
    sel = ref["eval_steps"] >= 1000
    Dref = ref["eval_steps"][sel] * ref_cfg["batch"]
    ax.plot(Dref, ref["per_out_loss"][sel][:, w], "k--", lw=1.2, label="multi-task")
    for s, col in zip(SEEDS, seed_cols):
        if (w, s) in solo:
            d = solo[(w, s)]
            m = d["eval_steps"] >= 1000
            ax.plot(d["eval_steps"][m] * ref_cfg["batch"], d["per_out_loss"][m][:, w],
                    color=col, lw=1.2, label=f"solo seed {s}")
    ax.axhline(CHANCE, color="gray", ls=":", lw=0.8)
    tag = " (parity)" if w in PARITY_WIRES else ""
    ax.set(xscale="log", yscale="log", title=f"wire {w}, depth {depths[w]}{tag}")
for ax in axes.flat[len(targets):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
fig.supxlabel("samples D"); fig.supylabel("per-output eval BCE")
plt.tight_layout()

## Transition times

First D where the target's eval BCE crosses below 0.2 nats (NaN = never).

In [ ]:
def transition_D(steps, loss, batch, thresh=0.2):
    below = np.flatnonzero(loss < thresh)
    return steps[below[0]] * batch if len(below) else np.nan


print(f"{'wire':>5s} {'depth':>6s} {'multi-task':>12s} " +
      " ".join(f"{'solo s' + str(s):>12s}" for s in SEEDS))
rows = []
for w in targets:
    tm = transition_D(ref["eval_steps"], ref["per_out_loss"][:, w], ref_cfg["batch"])
    ts = [transition_D(solo[(w, s)]["eval_steps"], solo[(w, s)]["per_out_loss"][:, w],
                       ref_cfg["batch"]) if (w, s) in solo else np.nan
          for s in SEEDS]
    rows.append((w, depths[w], tm, ts))
    print(f"{w:>5d} {depths[w]:>6d} {tm:>12,.0f} " +
          " ".join(f"{t:>12,.0f}" for t in ts))

fig, ax = plt.subplots(figsize=(6, 4.2))
for w, k, tm, ts in rows:
    mk = "*" if w in PARITY_WIRES else "o"
    ax.plot(k, tm, mk, color="k", ms=8, mfc="none")
    for t, col in zip(ts, seed_cols):
        ax.plot(k, t, mk, color=col, ms=7, alpha=0.8)
ax.set(yscale="log", xlabel="tap depth", ylabel="transition D (BCE < 0.2)",
       title="solo (color) vs multi-task (black open); * = parity")
plt.tight_layout()

## Notes

- Non-target outputs in solo runs stay untrained *at the head* (their head
  columns get no gradient), so their eval columns reflect init noise, not
  trunk knowledge — don't read incidental learning off them directly.
- The solo/multi-task transition-time ratio as a function of depth is the
  transfer measurement; parity wires isolate the no-low-degree-signal case
  (Barak et al. 2022). Expect strong seed dependence there — add seeds to
  `SEEDS` for those wires if the two disagree wildly.